# Step 01 — OSM extraction and preparation

Clip the regional OSM extract to the study area and prepare two layers.

| | |
|---|---|
| **Reads** | `data/input/regionalverband_area.gpkg`, `data/input/niedersachsen-*.osm.pbf` |
| **Writes** | `01_all_pois.gpkg` · `01_all_buildings_osm.gpkg` · `01_study_area_clipped.pbf` |
| **Needs** | `osmium` (system binary), `pyrosm` |
| **Runtime** | ~60 s |

**`01_all_pois.gpkg`** — the activity layer. Filtered to POIs that describe
something happening inside a building, each carrying `poi_use` (what happens
here) and `poi_role` (how it can be joined to a building).

**`01_all_buildings_osm.gpkg`** — every OSM building footprint, polygons only.
Not filtered by use: an unlabelled `building=yes` is still a real building.

Sections 1–2 clip. Section 3 prepares the buildings, 4–7 the POIs — in that
order because the role classification needs the building geometry. Section 8
writes both, so a failure above cannot leave one output newer than the other.

In [1]:
import os, shutil, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
# Do NOT use Path('..') here: a notebook's working directory is not necessarily
# its own folder. VS Code starts kernels in ${workspaceFolder} by default, so
# whenever the workspace is opened above this repo, '..' points somewhere else
# entirely and `import config` fails. Find the root by its marker file instead.
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError(
        'Cannot find the pipeline root (the folder containing config.py). '
        f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.'
    )
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# --- point GDAL/PROJ at this env's data files ---------------------------------
# Without these, pyogrio warns on every read and write and CRS lookups can fail
# outright. Must run before geopandas is imported; only sets what is missing.
_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import subprocess, tempfile, json, time

# --- find a working osmium ----------------------------------------------------
# Two separate traps, both silent.
#
# 1. A kernel started WITHOUT `conda activate` - which is how VS Code launches
#    one - has none of this environment's binaries on PATH. A bare
#    subprocess.run(['osmium', ...]) then dies with
#    `FileNotFoundError: [WinError 2] The system cannot find the file
#    specified`, naming neither osmium nor PATH.
#
# 2. `shutil.which('osmium')` returns `osmium.EXE` on Windows, upper-cased from
#    PATHEXT. osmium-tool 1.19 parses its own argv[0], does not recognise the
#    upper-case name, and exits 2 with `Unknown command or option 'osmium.EXE'`.
#    The identical binary spelled `osmium.exe` works.
#
# So candidates are verified by running them, never assumed. Preferring this
# env's copy also pins the version - base holds an older osmium that would
# otherwise win on PATH.
for _bin in (Path(sys.prefix) / 'Library' / 'bin',
             Path(sys.prefix) / 'Scripts',
             Path(sys.prefix) / 'bin'):
    if _bin.is_dir() and str(_bin) not in os.environ.get('PATH', ''):
        os.environ['PATH'] = str(_bin) + os.pathsep + os.environ.get('PATH', '')


def _find_osmium():
    cands = []
    for d in (Path(sys.prefix) / 'Library' / 'bin',
              Path(sys.prefix) / 'Scripts',
              Path(sys.prefix) / 'bin'):
        cands += [d / 'osmium.exe', d / 'osmium']
    found = shutil.which('osmium')
    if found:
        f = Path(found)
        cands += [f.with_suffix(f.suffix.lower()), f]
    for c in cands:
        if not c.is_file():
            continue
        try:
            r = subprocess.run([str(c), '--version'], capture_output=True, text=True)
        except OSError:
            continue
        if r.returncode == 0:
            return str(c), r.stdout.splitlines()[0]
    return None, None


OSMIUM, _osmium_version = _find_osmium()
if OSMIUM is None:
    raise RuntimeError(
        'No working osmium found. Section 2 needs it to clip the PBF. Looked in '
        f'{sys.prefix} and on PATH. Install it with:  '
        'conda install -c conda-forge osmium-tool'
    )

import pandas as pd
import geopandas as gpd
from pyrosm import OSM

from config import (
    STUDY_BOUNDARY_FILE, OSM_PBF_FILE,
    CLIPPED_PBF_FILE, ALL_POIS_FILE, ALL_BUILDINGS_OSM_FILE,
    OUTPUT_DIR, TARGET_CRS,
    POI_EXTRACT_FILTER, POI_EXTRA_ATTRIBUTES, POI_USE_SOURCES,
    EXCLUDE_AMENITIES, EXCLUDE_BUILDING_TYPES,
    ALLOWED_TOURISM_TYPES, ALLOWED_INFORMATION_TYPES, ALLOWED_LEISURE_TYPES,
    EXCLUDE_LIFECYCLE_PREFIXES, EXCLUDE_PLACEHOLDER_USES,
    POI_ANCILLARY_BUILDING_TAGS, POI_MIN_BUILDING_AREA_M2,
    POI_KEEP_TAG_COLS, POI_KEEP_DESC_COLS, POI_KEEP_ADDR_COLS,
    POI_DROP_META_COLS,
    BUILDING_KEEP_COLS, BUILDING_DROP_META_COLS,
)
from lib.checks import (
    require_file, require_non_empty, require_crs, require_unique, count_outside,
)
from lib.schema import fold_tags, osm_key, tidy_names, assert_no_empty_columns

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Root       :', ROOT_DIR)
print('osmium     :', _osmium_version, '|', OSMIUM)
print('Target CRS :', TARGET_CRS)

Root       : c:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL
osmium     : osmium version 1.19.1 | c:\Users\Mayur Patel\anaconda3\envs\capacity-final\Library\bin\osmium.exe
Target CRS : EPSG:25832


## 1. Input contract

Both inputs must exist before anything else runs — a missing PBF should fail
here in a second, not twenty minutes into an `osmium` call.

The boundary is dissolved once, in the target CRS, and reused for every
containment report below.

In [2]:
require_file(STUDY_BOUNDARY_FILE, 'study boundary')
require_file(OSM_PBF_FILE, 'OSM regional extract')

boundary = gpd.read_file(STUDY_BOUNDARY_FILE)
require_non_empty(boundary, 'boundary')
print(f'  ..  boundary CRS: {boundary.crs.to_string()}, {len(boundary)} feature(s)')

boundary_target = boundary.to_crs(TARGET_CRS)
boundary_geom   = boundary_target.geometry.union_all()
print(f'  ..  bounds (target CRS): {tuple(round(v) for v in boundary_geom.bounds)}')

  ok  regionalverband_area.gpkg (0.2 MB)
  ok  niedersachsen-260113.osm.pbf (478.2 MB)
  ok  boundary: 9 rows
  ..  boundary CRS: EPSG:25832, 9 feature(s)
  ..  bounds (target CRS): (567880, 5721966, 642410, 5854769)


## 2. Clip the PBF

`osmium` needs the polygon in WGS84 regardless of the boundary file's own CRS.

`-s complete_ways` keeps any way that touches the polygon *whole*, so a building
straddling the border survives intact rather than being cut into an invalid
geometry. The cost is a small overhang outside the boundary, which the
containment reports below quantify.

In [3]:
boundary_wgs = boundary.to_crs(epsg=4326)
geom_wgs     = boundary_wgs.geometry.union_all()

with tempfile.NamedTemporaryFile(mode='w', suffix='.geojson', delete=False) as f:
    json.dump({'type': 'Feature', 'geometry': geom_wgs.__geo_interface__, 'properties': {}}, f)
    poly_file = f.name
print(f'Clip polygon: {geom_wgs.geom_type}, '
      f'{len(geom_wgs.exterior.coords) if geom_wgs.geom_type == "Polygon" else "multi":,} vertices')

# Streamed line by line rather than captured: subprocess output goes to a file
# descriptor and ipykernel only redirects Python-level stdout, so a captured run
# prints NOTHING until it returns and looks like a hang.
print(f'Clipping {OSM_PBF_FILE.stat().st_size / 1e6:,.0f} MB with osmium '
      f'(expect ~5 s warm, up to a minute on a cold cache) ...', flush=True)
t0 = time.perf_counter()
proc = subprocess.Popen(
    [OSMIUM, 'extract',
     '-p', poly_file,
     '-s', 'complete_ways',
     '-o', str(CLIPPED_PBF_FILE),
     '--overwrite',
     str(OSM_PBF_FILE)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
tail = []
for line in proc.stdout:
    line = line.rstrip()
    if line:
        tail.append(line)
        print('   ', line, flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError('osmium extract failed (rc={}):\n{}'.format(rc, '\n'.join(tail[-15:])))

print(f'Clipped PBF: {CLIPPED_PBF_FILE.stat().st_size / 1e6:,.1f} MB '
      f'-> {CLIPPED_PBF_FILE.name}   [{time.perf_counter() - t0:.1f}s]')

Clip polygon: Polygon, 3,299 vertices
Clipping 478 MB with osmium (expect ~5 s warm, up to a minute on a cold cache) ...
Clipped PBF: 57.7 MB -> 01_study_area_clipped.pbf   [4.5s]


## 3. Buildings

A fresh `OSM` object is used per extraction: the reader is stateful, and reusing
one across `get_buildings()` / `get_pois()` has produced empty results.

Two things are removed.

**Non-polygon geometry.** A building is an area. OSM occasionally carries a
`building=*` tag on an unclosed way, which arrives here as a LineString — no
footprint, no area, nothing to match a building against.

**OSM edit metadata and contact details** — `version`, `changeset`, `visible`,
`website`, `phone`. Tags are folded into `tags`; the rest describe the edit
rather than the building and are dropped.

`building:levels` and `height` are kept: they are the only OSM inputs to a
volume.

In [4]:
print('Parsing the clipped PBF for buildings (expect ~40 s) ...', flush=True)
_t0 = time.perf_counter()
osm_bld = OSM(str(CLIPPED_PBF_FILE))
buildings = osm_bld.get_buildings()
print(f'  ..  parsed in {time.perf_counter() - _t0:.1f}s')

require_non_empty(buildings, 'osm_buildings')
buildings = buildings.to_crs(TARGET_CRS)
require_crs(buildings, TARGET_CRS, 'osm_buildings')
print('  ..  raw geometry types:', buildings.geometry.geom_type.value_counts().to_dict())

is_poly = buildings.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
if (~is_poly).any():
    print(f'  ..  dropped {(~is_poly).sum():,} non-polygon geometries: '
          f'{buildings.loc[~is_poly].geometry.geom_type.value_counts().to_dict()}')
buildings = buildings.loc[is_poly].reset_index(drop=True)
require_non_empty(buildings, 'osm_buildings (polygons)')
print(f'  ..  invalid geometries: {int((~buildings.geometry.is_valid).sum()):,}')

buildings, folded = fold_tags(buildings,
                              keep=BUILDING_KEEP_COLS + ['osm_type', 'id'],
                              meta=BUILDING_DROP_META_COLS)
print(f'  ..  folded {len(folded)} sparse tag columns into `tags`')

buildings = osm_key(buildings, 'bld_id')
buildings = buildings[
    ['bld_id', 'osm_type', 'id']
    + [c for c in BUILDING_KEEP_COLS if c in buildings.columns]
    + ['tags', 'geometry']
].rename(columns={'id': 'osm_id'})
buildings = tidy_names(buildings).reset_index(drop=True)

require_unique(buildings, 'bld_id', 'osm_buildings')
assert_no_empty_columns(buildings, 'osm_buildings')
bld_outside = count_outside(buildings, boundary_geom, 'osm_buildings')
print(f'  ok  buildings prepared: {len(buildings):,} rows, {len(buildings.columns)} columns')

Parsing the clipped PBF for buildings (expect ~40 s) ...
  ..  parsed in 23.8s
  ok  osm_buildings: 509,793 rows
  ok  osm_buildings: CRS EPSG:25832
  ..  raw geometry types: {'Polygon': 509757, 'MultiPolygon': 35, 'LineString': 1}
  ..  dropped 1 non-polygon geometries: {'LineString': 1}
  ok  osm_buildings (polygons): 509,792 rows
  ..  invalid geometries: 0
  ..  folded 0 sparse tag columns into `tags`
  ok  osm_buildings.bld_id: unique and non-null (509,792)
  ok  osm_buildings: 23 columns, none empty
  ..  osm_buildings: 0 of 509,792 rows outside boundary (0.00 %)
  ok  buildings prepared: 509,792 rows, 23 columns


## 4. POIs

**Which keys are extracted is set explicitly**, in `POI_EXTRACT_FILTER`.
`pyrosm`'s default is `amenity`, `shop` and `tourism` and nothing else, which
loses every workplace OSM describes with another key. In this region that is
1,525 `office` features of which only 27 also carry one of the three, plus
~310 `healthcare`-only and ~174 `club`-only features. Offices are occupied
buildings; missing 1,500 of them is a larger error for a capacity pipeline
than any amount of street furniture wrongly kept.

`healthcare` and `club` go through `extra_attributes` rather than the filter:
neither is in `pyrosm`'s tag configuration, so naming them in the filter alone
would match on them without promoting them to columns.

Line geometries are removed. OSM tags a handful of POIs onto ways that are not
areas - a path tagged `tourism=information`, a barrier tagged `amenity=*`. They
are neither a location nor a footprint.

Points and areas both stay. Roughly 36 % of OSM POIs are mapped as areas
(schools, supermarkets and sports centres drawn as their building outline), and
those footprints carry more information than a centroid does.

In [5]:
print('Parsing the clipped PBF for POIs (expect ~25 s) ...', flush=True)
_t0 = time.perf_counter()
osm_pois = OSM(str(CLIPPED_PBF_FILE))
pois = osm_pois.get_pois(custom_filter=POI_EXTRACT_FILTER,
                         extra_attributes=POI_EXTRA_ATTRIBUTES)
print(f'  ..  parsed in {time.perf_counter() - _t0:.1f}s')

require_non_empty(pois, 'pois')
_missing = [k for k in list(POI_EXTRACT_FILTER) + POI_EXTRA_ATTRIBUTES
            if k not in pois.columns]
if _missing:
    raise AssertionError(
        f'pyrosm returned no column for {_missing}. The extraction filter asked '
        'for these keys, so a missing column means the filter was ignored and '
        'every feature carrying only that key is absent from the layer.'
    )
print('  ..  extracted per key:',
      {k: int(pois[k].notna().sum())
       for k in list(POI_EXTRACT_FILTER) + POI_EXTRA_ATTRIBUTES})
n_pois_raw = len(pois)
pois = pois.to_crs(TARGET_CRS)
require_crs(pois, TARGET_CRS, 'pois')
print('  ..  raw geometry types:', pois.geometry.geom_type.value_counts().to_dict())

is_line = pois.geometry.geom_type.isin(['LineString', 'MultiLineString'])
if is_line.any():
    print(f'  ..  dropped {is_line.sum():,} line geometries')
pois = pois.loc[~is_line].reset_index(drop=True)
require_non_empty(pois, 'pois (points + areas)')
n_pois_geom = len(pois)

Parsing the clipped PBF for POIs (expect ~25 s) ...
  ..  parsed in 21.4s
  ok  pois: 79,609 rows
  ..  extracted per key: {'amenity': 52725, 'shop': 6175, 'tourism': 9639, 'office': 1525, 'craft': 644, 'leisure': 9291, 'healthcare': 938, 'club': 29}
  ok  pois: CRS EPSG:25832
  ..  raw geometry types: {'Point': 47035, 'Polygon': 32381, 'LineString': 69, 'MultiPolygon': 64, 'MultiLineString': 60}
  ..  dropped 129 line geometries
  ok  pois (points + areas): 79,480 rows


## 5. Keep only POIs that describe an indoor activity

**What.** Every POI is judged on its tags alone, and dropped if none of them
names something that happens inside a building.

**How.** Six rules, in `config.py`, each able to veto a row on its own:

| Column | Rule | Why this direction |
|---|---|---|
| `amenity` | drop 63 listed values | the useful values are open-ended; a keep-list would silently drop new ones |
| `building` | drop `roof`, `shed`, `hut`, `container`, `no` | not enterable structures |
| `tourism` | keep 8 listed values | nearly every other value is an outdoor feature |
| `information` | keep `office` | same |
| `leisure` | keep 16 listed values, **unless another activity key already gives a use** | mostly outdoor: 2,878 pitches, 1,955 playgrounds, 711 garden pools, 670 parks of 9,291 |
| lifecycle | drop values prefixed `disused:`, `abandoned:`, ... and the placeholders `construction`, `proposed` | whatever happened here, it does not happen now |

**Multi-value tags.** OSM separates multiple values with `;`, and `isin` is
exact-match, so `amenity=waste_basket;vending_machine` used to pass the
exclusion list untouched. Values are now split before matching, with
deliberately different quantifiers:

* an **exclusion** vetoes only when *every* token is excluded, so
  `amenity=theatre;parking` survives as a theatre rather than dying as a car
  park;
* a **keep-list** admits when *any* token qualifies, so
  `tourism=hotel;attraction` survives as a hotel.

**The `leisure` exception.** `leisure` is the one rule that is not a pure veto.
It is a secondary tag as often as a primary one - a restaurant with a beer
garden carries `leisure=garden` - so vetoing on it would drop features that
another key has already justified. A row is therefore only judged on `leisure`
when `leisure` is the *only* thing describing it. The amenity exclusion still
runs first and independently, so `amenity=parking` + `leisure=pitch` is dropped
either way.

`building` does **not** count as that other key. Letting it count readmitted 58
outdoor features on the strength of a bare `building=yes`, and because
`leisure` outranks `building` in the priority order their `poi_use` came out as
`pitch`, `stadium` or `playground`. A structure tag says the thing is a
building; it does not say what happens inside it.

Each other column vetoes independently, and that is the point: `amenity` is
OSM's primary use descriptor, so `amenity=shelter` disqualifies a feature even
when it also carries `building=yes`. Judging on any-signal-present instead would
readmit ~900 shelters, toilets and bike shelters through their structure tag.

**Why.** Roughly three quarters of what OSM calls a POI is street furniture,
parking or open-air infrastructure - benches, waste baskets, hunting stands,
post boxes, viewpoints. Each of those sits inside some building polygon and
would either say nothing about it or actively mislabel it.

Three values were reviewed against this region and settled against the list
used previously: `grave_yard` is **kept** (78 of its 153 contain a real building
- a chapel or hall), `bus_station` is **dropped** (the 15 here are paved
forecourts of 571-4,136 m2; the only one tagged `building=yes` is 13.3 m2), and
`leisure=stadium`/`water_park` are **dropped** because the occupancy they
describe is not inside a building.

This is the only place a POI is dropped, and it drops on the tag alone - never
on how many buildings happen to sit inside it.

In [6]:
def _tokens(v):
    """Split an OSM multi-value tag into its parts, `;`-separated."""
    return [t.strip() for t in str(v).split(';') if t.strip()]


def _all_tokens_in(col, values):
    """True where every token of the value is in `values` (False where null).

    Used by the exclusion rules. `all` rather than `any` so a value pairing a
    real use with a dropped one - `amenity=theatre;parking` - is kept.
    """
    values = set(values)
    return col.map(
        lambda v: all(t in values for t in _tokens(v)) if pd.notna(v) else False
    ).astype(bool)


def _any_token_in(col, values):
    """True where any token of the value is in `values` (False where null).

    Used by the keep-lists, so `tourism=hotel;attraction` is admitted.
    """
    values = set(values)
    return col.map(
        lambda v: any(t in values for t in _tokens(v)) if pd.notna(v) else False
    ).astype(bool)


def is_informative(v):
    """False for a null, a lifecycle-prefixed value or a placeholder."""
    if pd.isna(v):
        return False
    return any(
        t and not t.startswith(EXCLUDE_LIFECYCLE_PREFIXES)
        and t not in EXCLUDE_PLACEHOLDER_USES
        for t in _tokens(v)
    )


def _has_other_use(d):
    """True where an ACTIVITY key other than `leisure` already describes a use.

    `building` is excluded deliberately, though it is a use source. Counting it
    here readmitted 58 outdoor features - pitches, stadiums, playgrounds - on
    the strength of a bare `building=yes`, and `poi_use` then resolved to the
    outdoor value because `leisure` outranks `building`. A structure tag says
    the thing is a building; it does not say what happens inside it, which is
    the one question this rule is asking. The cost is ~15 genuine indoor halls
    tagged only `leisure=water_park`/`horse_riding` on a building outline; add
    those values to ALLOWED_LEISURE_TYPES if that trade looks wrong for a
    region.
    """
    cols = [c for c in POI_USE_SOURCES
            if c not in ('leisure', 'building') and c in d.columns]
    return pd.concat([d[c].map(is_informative) for c in cols], axis=1).any(axis=1)


# Each rule is a callable, not a precomputed mask. Building all the masks up
# front would evaluate every rule against the UNFILTERED frame: the counts for
# the later rules would double-count rows an earlier rule already removed, and
# applying a stale boolean to a shrunken frame triggers a reindex.
RULES = (
    ('amenity exclusion',
     lambda d: ~_all_tokens_in(d['amenity'], EXCLUDE_AMENITIES)),
    ('building exclusion',
     lambda d: ~_all_tokens_in(d['building'], EXCLUDE_BUILDING_TYPES)),
    ('tourism keep-list',
     lambda d: d['tourism'].isna() | _any_token_in(d['tourism'], ALLOWED_TOURISM_TYPES)),
    ('information keep-list',
     lambda d: d['information'].isna() | _any_token_in(d['information'], ALLOWED_INFORMATION_TYPES)),
    # Not a pure veto - see section 5. Only judged when `leisure` is the only
    # key describing the feature.
    ('leisure keep-list',
     lambda d: d['leisure'].isna()
               | _any_token_in(d['leisure'], ALLOWED_LEISURE_TYPES)
               | _has_other_use(d)),
    # Every use-bearing tag is dead or a placeholder, so nothing happens here.
    ('lifecycle / placeholder',
     lambda d: pd.concat(
         [d[c].map(is_informative) for c in POI_USE_SOURCES if c in d.columns],
         axis=1).any(axis=1)),
)

n_before = len(pois)
for label, rule in RULES:
    keep = rule(pois)
    print(f'  ..  {label:<24} -{int((~keep).sum()):>7,}')
    pois = pois.loc[keep]

pois = pois.reset_index(drop=True)
require_non_empty(pois, 'pois (filtered)')
print(f'  ok  kept {len(pois):,} of {n_before:,} ({100 * len(pois) / n_before:.1f} %)')

  ..  amenity exclusion        - 43,448
  ..  building exclusion       -     55
  ..  tourism keep-list        -  8,391
  ..  information keep-list    -      6
  ..  leisure keep-list        -  8,180
  ..  lifecycle / placeholder  -      1
  ok  pois (filtered): 19,399 rows
  ok  kept 19,399 of 79,480 (24.4 %)


## 6. `poi_role` — how each POI can be joined to a building

Area POIs are not one kind of thing, and joining them uniformly is wrong either
way:

* **collapsing areas to points** reduces a 370,000 m² university campus to one
  arbitrary shed, and can drop a centroid into an unrelated neighbour;
* **joining every area to every building it touches** pulls in neighbours whose
  wall merely abuts the site boundary — measured at ~6 % spurious pairs.

| `poi_role` | Meaning | Join by |
|---|---|---|
| `point` | node POI | the building containing it |
| `footprint` | area covering one building, or none in OSM | largest intersection area, 1:1 |
| `site` | area covering 2+ real buildings (school, hospital, campus) | all contained buildings, carrying `poi_id` as a site key |

`site` marks an area covering several buildings — a hospital campus really is
several hospital buildings — and `poi_id` identifies which buildings belong to
the same site.

Counting raw footprints would make a school with three garages look like a
four-building site, so ancillary structures and anything below
`POI_MIN_BUILDING_AREA_M2` are ignored. Buildings are matched on
`representative_point` **within** the area, not `intersects`, for the reason
above.

An area with **no** building inside it is classed `footprint`, not discarded.
That emptiness is a fact about OSM's coverage, not about reality. `n_osm_bld`
records what was found and is a hint, not a verdict.

In [7]:
areas_mask = pois.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
areas = pois.loc[areas_mask, ['geometry']].copy()
areas['_aix'] = range(len(areas))

bld_real = buildings.loc[
    ~buildings['building'].isin(POI_ANCILLARY_BUILDING_TAGS)
    & (buildings.geometry.area >= POI_MIN_BUILDING_AREA_M2)
]
print(f'  ..  buildings counted as real: {len(bld_real):,} of {len(buildings):,} '
      f'({100 * len(bld_real) / len(buildings):.1f} %)')

if areas.empty:
    n_bld = pd.Series(dtype='int64')
else:
    pts = gpd.GeoDataFrame(geometry=bld_real.geometry.representative_point(),
                           crs=bld_real.crs)
    hit = gpd.sjoin(pts, areas, how='inner', predicate='within')
    n_bld = areas['_aix'].map(hit.groupby('_aix').size()).fillna(0).astype(int)

pois['n_osm_bld'] = 0
pois.loc[areas.index, 'n_osm_bld'] = n_bld.values
pois['geom_kind'] = pois.geometry.geom_type.map(
    {'Point': 'point', 'Polygon': 'area', 'MultiPolygon': 'area'})
pois['area_m2'] = 0.0
pois.loc[areas.index, 'area_m2'] = pois.loc[areas.index].geometry.area.round(1)

pois['poi_role'] = 'point'
pois.loc[areas_mask, 'poi_role'] = 'footprint'
pois.loc[areas_mask & (pois['n_osm_bld'] >= 2), 'poi_role'] = 'site'

print()
print('  ..  poi_role:')
for role, cnt in pois['poi_role'].value_counts().items():
    print(f'        {role:<10} {cnt:>7,}')
print(f'  ..  areas with no OSM building: '
      f'{int((areas_mask & (pois["n_osm_bld"] == 0)).sum()):,}')

  ..  buildings counted as real: 386,014 of 509,792 (75.7 %)

  ..  poi_role:
        point       11,255
        footprint    7,430
        site           714
  ..  areas with no OSM building: 686


## 7. `poi_use` and the schema

`poi_use` collapses the use-bearing tag columns into one answer: *what happens
here*. `poi_use_tag` records which tag it came from, because that distinction
matters - `shop=bakery` and `amenity=bakery` should not be classified by the
same rule.

Priority is `POI_USE_SOURCES`: `amenity > shop > office > craft > healthcare >
tourism > club > leisure > building`. `amenity` is OSM's primary use
descriptor, so `amenity=shelter` wins over `building=yes` on the same feature;
`building` is last because it describes the structure, not the activity inside
it. `club` outranks `leisure` because a sports club carrying `club=sport` +
`leisure=pitch` is a clubhouse, and `sport` describes it better than `pitch`.

`office=yes` and `shop=yes` resolve to the **key**, not to the literal `yes`:
that a use exists but is unspecified is worth recording, `yes` on its own is
not. Without this, widening the extraction filter would have grown
`poi_use='yes'` from 43 rows to 125.

A value that is not informative - lifecycle-prefixed or a placeholder - is
skipped rather than taken, so `amenity=disused:restaurant` + `shop=bakery`
resolves to `bakery` instead of to a restaurant that closed. Section 5 has
already dropped the rows where *nothing* informative was left, so `poi_use`
cannot come out null.

The schema reduction is the same lossless fold used for the buildings: every
dropped tag is folded into `tags`, which is where `capacity`, `capacity:beds`,
`beds`, `seats` and `level` live.

In [8]:
pois['poi_use'] = pd.Series(pd.NA, index=pois.index, dtype='object')
pois['poi_use_tag'] = pd.Series(pd.NA, index=pois.index, dtype='object')
for _col in POI_USE_SOURCES:
    if _col not in pois.columns:
        continue
    # `is_informative`, not `notna`: a dead or placeholder value must not win
    # the slot and block a lower-priority column that has a real answer.
    _fill = pois['poi_use'].isna() & pois[_col].map(is_informative)
    # `office=yes` says a use exists but not which one. Resolving it to the KEY
    # rather than the literal 'yes' keeps the one fact it does carry - this is
    # an office - instead of a value that means nothing on its own. Widening
    # the extraction filter would otherwise have grown `poi_use='yes'` from 43
    # rows to 125.
    _val = pois.loc[_fill, _col]
    pois.loc[_fill, 'poi_use'] = _val.where(_val.astype(str).str.strip() != 'yes', _col)
    pois.loc[_fill, 'poi_use_tag'] = _col
print('  ..  poi_use resolved from:', pois['poi_use_tag'].value_counts().to_dict())
_n_null = int(pois['poi_use'].isna().sum())
if _n_null:
    raise AssertionError(
        f'{_n_null:,} POIs have no informative use tag, but section 5 should '
        'already have dropped every such row. The two rules have drifted apart.'
    )
print(f'  ..  poi_use null: {_n_null:,} of {len(pois):,}')

derived = ['poi_use', 'poi_use_tag', 'poi_role', 'geom_kind', 'area_m2', 'n_osm_bld']
keep = (POI_KEEP_DESC_COLS + POI_KEEP_TAG_COLS + POI_KEEP_ADDR_COLS
        + derived + ['osm_type', 'id'])

pois, folded = fold_tags(pois, keep=keep, meta=POI_DROP_META_COLS)
print(f'  ..  folded {len(folded)} sparse tag columns into `tags`')

pois = osm_key(pois, 'poi_id')
pois = pois[
    ['poi_id', 'osm_type', 'id'] + derived
    + [c for c in POI_KEEP_DESC_COLS if c in pois.columns]
    + [c for c in POI_KEEP_TAG_COLS if c in pois.columns]
    + [c for c in POI_KEEP_ADDR_COLS if c in pois.columns]
    + ['tags', 'geometry']
].rename(columns={'id': 'osm_id'})
pois = tidy_names(pois).reset_index(drop=True)

require_unique(pois, 'poi_id', 'pois')
assert_no_empty_columns(pois, 'pois')
pois_outside = count_outside(pois, boundary_geom, 'pois')
print(f'  ok  POIs prepared: {len(pois):,} rows, {len(pois.columns)} columns')

  ..  poi_use resolved from: {'amenity': 9173, 'shop': 6067, 'office': 1496, 'tourism': 1104, 'leisure': 973, 'craft': 573, 'club': 7, 'building': 4, 'healthcare': 2}
  ..  poi_use null: 0 of 19,399
  ..  folded 98 sparse tag columns into `tags`
  ok  pois.poi_id: unique and non-null (19,399)
  ok  pois: 31 columns, none empty
  ..  pois: 0 of 19,399 rows outside boundary (0.00 %)
  ok  POIs prepared: 19,399 rows, 31 columns


## 8. Write both outputs

Both layers are final before either is written. Each file is unlinked first — a
GeoPackage write *appends*, so a stale layer from an earlier run would otherwise
survive next to the new one.

In [9]:
for path, frame, layer in (
    (ALL_POIS_FILE, pois, 'pois'),
    (ALL_BUILDINGS_OSM_FILE, buildings, 'buildings'),
):
    if path.exists():
        try:
            path.unlink()
        except PermissionError as e:
            raise RuntimeError(
                f'{path.name} is locked by another process, so it cannot be '
                'replaced. QGIS holds a GeoPackage open for as long as the '
                'layer is loaded - remove the layer (or close the project) and '
                f'run this cell again. Original error: {e}'
            ) from None
    print(f'Writing {len(frame):,} rows to {path.name} ...', flush=True)
    _t0 = time.perf_counter()
    frame.to_file(path, layer=layer, driver='GPKG')
    print(f'Wrote {len(frame):>9,} rows, {len(frame.columns):>2} cols, '
          f'{path.stat().st_size / 1e6:>6,.1f} MB  -> {path.name} : {layer}'
          f'   [{time.perf_counter() - _t0:.1f}s]')

Writing 19,399 rows to 01_all_pois.gpkg ...
Wrote    19,399 rows, 31 cols,    9.4 MB  -> 01_all_pois.gpkg : pois   [0.3s]
Writing 509,792 rows to 01_all_buildings_osm.gpkg ...
Wrote   509,792 rows, 23 cols,  160.7 MB  -> 01_all_buildings_osm.gpkg : buildings   [4.8s]


## 9. QGIS checkpoint

Load both outputs plus `regionalverband_area.gpkg` and confirm:

1. **Coverage** — both layers blanket the whole region, with no empty
   rectangular holes. A hole means the PBF did not cover the boundary.
2. **Alignment** — buildings sit on a basemap correctly. If everything is in
   the Atlantic near (0, 0), a CRS was mis-assigned rather than reprojected.
3. **Overhang** — the "outside boundary" counts printed above should be small.
4. **Plausibility** — style `pois` by `poi_use`: shops, schools, restaurants and
   surgeries, dense in town centres. If you still see benches and parking, the
   filter in section 5 did not run.
5. **Roles** — filter `poi_role = 'site'`: each should be a campus or complex,
   not a single building. Then filter `poi_role = 'footprint' AND n_osm_bld = 0`
   and check a few against a basemap.

`pois` holds points and areas in one layer, so QGIS lists it as two entries,
`pois (Point)` and `pois (Polygon)`. That is a display convention for a generic
GEOMETRY layer, not two layers — load both, or you are looking at a subset.
`buildings` is polygon-only and appears once.

In [10]:
print('STEP 01 SUMMARY')
print('POI funnel:')
print(f'  extracted from PBF    : {n_pois_raw:>8,}')
print(f'  after dropping lines  : {n_pois_geom:>8,}  (-{n_pois_raw - n_pois_geom:,})')
print(f'  after semantic filter : {len(pois):>8,}  (-{n_pois_geom - len(pois):,})')
print(f'  kept                  : {100 * len(pois) / n_pois_raw:>7.1f} % of extracted')
print()
print(f'  POIs      : {len(pois):>9,} rows  {len(pois.columns):>2} cols  '
      f'({len(pois_outside):,} outside boundary)')
print(f'  Buildings : {len(buildings):>9,} rows  {len(buildings.columns):>2} cols  '
      f'({len(bld_outside):,} outside boundary)')
print()
print('POI role x geometry:')
print(pd.crosstab(pois['poi_role'], pois['geom_kind']).to_string())
print()
print('poi_use came from:')
print(pois['poi_use_tag'].value_counts(dropna=False).to_string())
print()
print('Top 15 poi_use values:')
print(pois['poi_use'].value_counts().head(15).to_string())
print()
print('Site POIs by real OSM building count:')
site = pois.loc[pois['poi_role'] == 'site', 'n_osm_bld']
if len(site):
    print(pd.cut(site, [1, 2, 5, 20, 100, 10**6],
                 labels=['2', '3-5', '6-20', '21-100', '>100'])
          .value_counts().sort_index().to_string())
print()
print('Buildings with a volume input:')
for c in ('building_levels', 'height'):
    if c in buildings.columns:
        n = int(buildings[c].notna().sum())
        print(f'  {c:<16} {n:>8,}  ({100 * n / len(buildings):5.1f} %)')
print()
print('Load in QGIS:')
for p in (STUDY_BOUNDARY_FILE, ALL_POIS_FILE, ALL_BUILDINGS_OSM_FILE):
    print('  ', p)

STEP 01 SUMMARY
POI funnel:
  extracted from PBF    :   79,609
  after dropping lines  :   79,480  (-129)
  after semantic filter :   19,399  (-60,081)
  kept                  :    24.4 % of extracted

  POIs      :    19,399 rows  31 cols  (0 outside boundary)
  Buildings :   509,792 rows  23 cols  (0 outside boundary)

POI role x geometry:
geom_kind  area  point
poi_role              
footprint  7430      0
point         0  11255
site        714      0

poi_use came from:
poi_use_tag
amenity       9173
shop          6067
office        1496
tourism       1104
leisure        973
craft          573
club             7
building         4
healthcare       2

Top 15 poi_use values:
poi_use
restaurant          1142
place_of_worship     879
sports_centre        708
kindergarten         655
school               591
fast_food            552
fire_station         523
hairdresser          521
supermarket          502
doctors              490
clothes              481
bakery               457
social